# 🔢 Count Models — Poisson, Negative Binomial & ZINB
**ISLP Chapter 4 (Extended) · Pattern Recognition for the Rest of Us**

> When your outcome is a count — number of ER visits, insurance claims, website clicks, product defects — linear regression is the wrong tool. Count models respect the discrete, non-negative, often skewed nature of count data.

---
### What you'll learn
- Why linear regression fails for count outcomes
- **Poisson regression** — the foundation of count modeling
- **Negative Binomial regression** — when counts are overdispersed
- **Zero-Inflated models (ZINB/ZIP)** — when there are too many zeros
- Interpreting rate ratios (IRR)
- Model diagnostics and choosing between models

### Real-world use cases
Customer support tickets, hospital admissions, insurance claims, website visits, manufacturing defects, bike rentals, species counts in ecology

### Dataset
We'll use a **bike sharing dataset** (hourly bike rentals) and a synthetic **insurance claims dataset**

### Time
~70 minutes

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install statsmodels -q  # usually pre-installed in Colab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})
print("✓ Setup complete")

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)
DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'
print(f"✓ Data directory: {DATA_DIR}")

In [ ]:
# ── Generate datasets (no external URL needed) ──────────────────────────────
np.random.seed(42)

# ── Dataset 1: Bike rentals (hourly, count outcome) ──
n = 500
hour        = np.random.randint(0, 24, n)
temp        = np.random.uniform(0, 1, n)       # normalized temperature
humidity    = np.random.uniform(0, 1, n)       # normalized humidity
windspeed   = np.random.uniform(0, 0.8, n)
workingday  = np.random.binomial(1, 0.68, n)
season      = np.random.choice([1,2,3,4], n)

# True log-linear relationship
log_mu = (3.5
          + 0.8 * temp
          - 0.4 * humidity
          - 0.3 * windspeed
          + 0.3 * workingday
          + 0.2 * (season == 2).astype(float)
          + 0.4 * (season == 3).astype(float)
          + 0.15 * np.sin(2 * np.pi * hour / 24)
          + 0.6 * np.cos(2 * np.pi * (hour - 8) / 24))
bikes = pd.DataFrame({
    'count':    np.random.poisson(np.exp(log_mu)),
    'hour':     hour,
    'temp':     temp.round(3),
    'humidity': humidity.round(3),
    'windspeed':windspeed.round(3),
    'workingday': workingday,
    'season':   season
})

# ── Dataset 2: Insurance claims (overdispersed, many zeros) ──
n2 = 800
age         = np.random.randint(18, 75, n2)
exposure    = np.random.uniform(0.5, 1.0, n2)   # years of coverage
male        = np.random.binomial(1, 0.5, n2)
smoker      = np.random.binomial(1, 0.2, n2)
urban       = np.random.binomial(1, 0.6, n2)

log_mu2 = (-2.5
            + 0.03 * age
            + 0.4 * male
            + 0.8 * smoker
            + 0.3 * urban
            + np.log(exposure))   # offset for exposure time

# Negative binomial (overdispersed)
mu2  = np.exp(log_mu2)
alpha = 2.0  # overdispersion
r     = 1 / alpha
p_nb  = r / (r + mu2)
claims = np.random.negative_binomial(r, p_nb, n2)

# Add zero-inflation (some people never claim regardless of risk)
zero_inflate = np.random.binomial(1, 0.35, n2)  # 35% structural zeros
claims = claims * (1 - zero_inflate)

insurance = pd.DataFrame({
    'claims':   claims,
    'age':      age,
    'exposure': exposure.round(3),
    'male':     male,
    'smoker':   smoker,
    'urban':    urban
})

print(f"Bike rentals: {bikes.shape}  |  Mean count: {bikes['count'].mean():.1f}  |  Max: {bikes['count'].max()}")
print(f"Insurance:    {insurance.shape}  |  Zero rate: {(insurance['claims']==0).mean():.1%}  |  Mean claims: {insurance['claims'].mean():.2f}")

## 📐 Part 1 — Why Linear Regression Fails for Count Data

Count outcomes violate several key assumptions of OLS:

1. **Non-negative** — OLS can predict negative counts (impossible)
2. **Integer-valued** — OLS predicts continuous values
3. **Heteroscedastic** — variance in counts grows with the mean (for Poisson: Var = Mean)
4. **Right-skewed** — especially with rare events, the distribution is not normal
5. **Non-linear relationships** — a one-unit change in X has a *multiplicative* effect on counts, not additive

The fix: model **log(μ)** as a linear function of predictors:
```
log(E[Y|X]) = β₀ + β₁X₁ + β₂X₂ + ...
```
This is the **log link** — it ensures predictions are always positive and models multiplicative effects.

In [ ]:
# Show why OLS fails on count data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Distribution of counts — clearly not normal
axes[0].hist(bikes['count'], bins=40, color='#1e5fa8', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Bike Rentals\n(right-skewed, discrete, non-negative)')
axes[0].set_xlabel('Count'); axes[0].set_ylabel('Frequency')

# 2. OLS predicts negative counts
from sklearn.linear_model import LinearRegression
X_ols = bikes[['temp','humidity','windspeed','workingday']].values
y_ols = bikes['count'].values
ols   = LinearRegression().fit(X_ols, y_ols)
y_pred_ols = ols.predict(X_ols)
neg_pct = (y_pred_ols < 0).mean() * 100

axes[1].scatter(y_pred_ols, y_ols, alpha=0.3, s=15, color='#e85d20')
axes[1].axvline(0, color='red', lw=2, ls='--', label=f'Zero line\n({neg_pct:.1f}% predictions < 0)')
axes[1].set_xlabel('OLS Predicted Count')
axes[1].set_ylabel('Actual Count')
axes[1].set_title(f'OLS: {neg_pct:.1f}% of predictions are negative\n(impossible for count data!)')
axes[1].legend(fontsize=9)

# 3. Variance ≠ constant — grows with mean (Poisson property)
bins = pd.cut(bikes['count'], bins=10)
grp  = bikes.groupby(bins, observed=True)['count'].agg(['mean','var']).dropna()
axes[2].scatter(grp['mean'], grp['var'], color='#1a7a45', s=60, zorder=3)
x_line = np.linspace(grp['mean'].min(), grp['mean'].max(), 100)
axes[2].plot(x_line, x_line, color='#e85d20', lw=1.5, ls='--', label='Var = Mean (Poisson)')
axes[2].set_xlabel('Bin Mean'); axes[2].set_ylabel('Bin Variance')
axes[2].set_title('Variance grows with mean\n(violates OLS homoscedasticity assumption)')
axes[2].legend()

plt.suptitle('Why OLS Fails for Count Data', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 📊 Part 2 — Poisson Regression

The **Poisson distribution** models the probability of k events occurring in a fixed interval:
```
P(Y = k) = (μᵏ × e^{-μ}) / k!
```

Key property: **Mean = Variance = μ** (equidispersion)

**Poisson regression** assumes:
```
log(μᵢ) = β₀ + β₁X₁ᵢ + β₂X₂ᵢ + ... + βₚXₚᵢ
```

**Interpreting coefficients:**  
Exponentiating gives the **Incidence Rate Ratio (IRR)**:
- eᵝʲ = multiplicative change in the expected count per unit increase in Xⱼ
- e^0.3 ≈ 1.35 → a 35% increase in the expected count
- e^{-0.2} ≈ 0.82 → an 18% decrease in the expected count

**Offset terms:** When observations have different exposure times (person-years, hours, miles), add log(exposure) as an offset — it enters the model with coefficient fixed at 1.

In [ ]:
# Fit Poisson regression
poisson_model = smf.poisson(
    'count ~ temp + humidity + windspeed + workingday + C(season)',
    data=bikes
).fit(disp=False)

print(poisson_model.summary())

In [ ]:
# Extract and interpret coefficients as IRR (Incidence Rate Ratios)
coefs = poisson_model.params
irr   = np.exp(coefs)
ci    = np.exp(poisson_model.conf_int())
pvals = poisson_model.pvalues

summary = pd.DataFrame({
    'Coefficient (β)': coefs.round(4),
    'IRR (exp(β))':    irr.round(4),
    'IRR 95% CI Low':  ci[0].round(4),
    'IRR 95% CI High': ci[1].round(4),
    'p-value':         pvals.round(4),
    'Significant':     pvals < 0.05
})
print("=== Poisson Regression — Incidence Rate Ratios ===\n")
print(summary.to_string())
print("\n📌 Interpretation examples:")
print(f"  temp   IRR = {irr['temp']:.3f} → each unit increase in normalized temp multiplies expected rentals by {irr['temp']:.3f} (+{(irr['temp']-1)*100:.0f}%)")
print(f"  humid  IRR = {irr['humidity']:.3f} → each unit increase in humidity multiplies rentals by {irr['humidity']:.3f} (-{(1-irr['humidity'])*100:.0f}%)")
print(f"  workingday IRR = {irr['workingday']:.3f} → working days have {(irr['workingday']-1)*100:.0f}% more rentals than non-working days")

In [ ]:
# Visualize: actual vs predicted + IRR plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
y_pred_pois = poisson_model.predict()
axes[0].scatter(y_pred_pois, bikes['count'], alpha=0.3, s=15, color='#1e5fa8')
max_val = max(y_pred_pois.max(), bikes['count'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', lw=1.5, alpha=0.5, label='Perfect prediction')
axes[0].set_xlabel('Predicted Count (Poisson)')
axes[0].set_ylabel('Actual Count')
axes[0].set_title(f'Poisson: Actual vs Predicted\nPseudo-R² = {poisson_model.prsquared:.3f}')
axes[0].legend()

# IRR forest plot
coef_plot = summary.drop('Intercept', errors='ignore')
coef_plot = coef_plot[coef_plot.index.str.contains('temp|humid|wind|working|season')]
y_pos = range(len(coef_plot))
colors_irr = ['#1a7a45' if v > 1 else '#e85d20' for v in coef_plot['IRR (exp(β))']]

axes[1].barh(list(coef_plot.index), coef_plot['IRR (exp(β))'] - 1,
             left=1, color=colors_irr, alpha=0.7, edgecolor='white', height=0.5)
axes[1].errorbar(coef_plot['IRR (exp(β))'],
                 range(len(coef_plot)),
                 xerr=[coef_plot['IRR (exp(β))'] - coef_plot['IRR 95% CI Low'],
                       coef_plot['IRR 95% CI High'] - coef_plot['IRR (exp(β))']],
                 fmt='none', color='#1a1d23', capsize=4, lw=1.5)
axes[1].axvline(1, color='black', lw=1.5, ls='--', label='IRR = 1 (no effect)')
axes[1].set_xlabel('Incidence Rate Ratio (IRR)')
axes[1].set_title('Poisson IRR with 95% CI\n(green > 1 = increases count, red < 1 = decreases)')
axes[1].legend()

plt.tight_layout()
plt.show()

## ⚠️ Part 3 — Overdispersion: When Poisson Is Not Enough

The Poisson assumption **Var(Y) = E(Y)** is often violated in practice. When:
```
Var(Y) > E(Y)   →   overdispersion
Var(Y) < E(Y)   →   underdispersion (rare)
```

**Overdispersion** is extremely common. It happens when:
- There is unobserved heterogeneity (some people are just more claim-prone)
- Events are not independent (one claim triggers awareness of others)
- Data is a mixture of different populations

**Consequences of ignoring overdispersion:**
- Standard errors are too small → p-values too small → false discoveries
- Confidence intervals are too narrow → overconfident predictions

**Test for overdispersion:**  
Pearson chi-squared / degrees of freedom. If >> 1, you have overdispersion.

In [ ]:
# Fit Poisson on insurance claims and check for overdispersion
ins_pois = smf.poisson(
    'claims ~ age + male + smoker + urban + np.log(exposure)',
    data=insurance
).fit(disp=False)

# Overdispersion test: Pearson χ²/df >> 1 indicates overdispersion
pearson_resid = ins_pois.resid_pearson
dispersion = (pearson_resid**2).sum() / ins_pois.df_resid

print("=== Overdispersion Diagnostics ===\n")
print(f"Poisson assumption: Var(Y) = E(Y)  →  dispersion ratio = 1.0")
print(f"Observed dispersion ratio:  {dispersion:.2f}")
print(f"\nDispersion ratio >> 1  →  {'OVERDISPERSED — use Negative Binomial!' if dispersion > 1.5 else 'OK — Poisson may be fine'}")

# Visual check: variance vs mean by group
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = pd.cut(insurance['claims'], bins=8)
grp  = insurance.groupby(bins, observed=True)['claims'].agg(['mean','var']).dropna()
axes[0].scatter(grp['mean'], grp['var'], color='#1e5fa8', s=60, zorder=3, label='Observed')
x_line = np.linspace(0, grp['mean'].max(), 100)
axes[0].plot(x_line, x_line,           color='#1a7a45', lw=2, ls='--', label='Poisson (Var=Mean)')
axes[0].plot(x_line, x_line + 2*x_line**2, color='#e85d20', lw=2, ls='--', label='NegBin (Var>Mean)')
axes[0].set_xlabel('Mean Claims'); axes[0].set_ylabel('Variance')
axes[0].set_title(f'Variance vs Mean — Dispersion ratio = {dispersion:.1f}\n(above Poisson line → overdispersed)')
axes[0].legend()

# Distribution of claims
axes[1].hist(insurance['claims'], bins=range(0, insurance['claims'].max()+2),
             color='#1e5fa8', edgecolor='white', alpha=0.8, density=True)
# Overlay expected Poisson
mu_hat = insurance['claims'].mean()
k = np.arange(0, insurance['claims'].max()+1)
axes[1].plot(k, stats.poisson.pmf(k, mu_hat), 'ro-', ms=5, lw=1.5,
             label=f'Poisson(μ={mu_hat:.2f})', alpha=0.8)
axes[1].set_xlabel('Number of Claims')
axes[1].set_ylabel('Density')
axes[1].set_title('Claim Distribution — excess zeros and heavy tail\nvs expected Poisson')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nObserved zero rate: {(insurance['claims']==0).mean():.1%}")
print(f"Poisson expected zeros: {stats.poisson.pmf(0, mu_hat):.1%}")
print("📌 Far more zeros than Poisson predicts — zero-inflation present")

## 📈 Part 4 — Negative Binomial Regression

The **Negative Binomial** distribution adds a dispersion parameter α to Poisson:
```
Var(Y) = μ + α·μ²
```

When α → 0: NB → Poisson. Larger α = more overdispersion.

NB is a **Poisson-Gamma mixture**: imagine each observation has its own μᵢ drawn from a Gamma distribution. This accounts for unobserved heterogeneity — some people are inherently more claim-prone than average.

**Model:**
```
log(μᵢ) = β₀ + β₁X₁ + ... + offset
```
Same link function as Poisson, same IRR interpretation, but with an estimated α parameter.

In [ ]:
# Fit Negative Binomial
nb_model = smf.negativebinomial(
    'claims ~ age + male + smoker + urban + np.log(exposure)',
    data=insurance
).fit(disp=False)

print(nb_model.summary())
print(f"\nEstimated dispersion (alpha): {nb_model.params['alpha']:.4f}")
print("→ Var(Y) = μ + {:.4f}·μ²   (alpha=0 would be Poisson)".format(nb_model.params['alpha']))

In [ ]:
# Compare Poisson vs NB: standard errors tell the story
pois_se = ins_pois.bse.drop('Intercept', errors='ignore')
nb_se   = nb_model.bse.drop(['Intercept','alpha'], errors='ignore')

common = pois_se.index.intersection(nb_se.index)
se_df  = pd.DataFrame({'Poisson SE': pois_se[common], 'NegBin SE': nb_se[common]})
se_df['SE ratio (NB/Pois)'] = (nb_se[common] / pois_se[common]).round(3)

print("=== Standard Error Comparison: Poisson vs Negative Binomial ===\n")
print(se_df.round(4).to_string())
print("\n📌 NB standard errors are larger — Poisson was overconfident!")
print("   Poisson underestimated uncertainty because it ignored overdispersion.")
print("   Some Poisson 'significant' predictors may not be significant under NB.")

# Model comparison: AIC
print(f"\n=== Model Fit (lower AIC = better) ===")
print(f"  Poisson AIC:          {ins_pois.aic:.2f}")
print(f"  Negative Binomial AIC: {nb_model.aic:.2f}  ({'✓ Better' if nb_model.aic < ins_pois.aic else '✗ Worse'})")

## 0️⃣ Part 5 — Zero-Inflated Models (ZIP and ZINB)

Sometimes there are **too many zeros** even for a Negative Binomial. This happens when zeros come from two different processes:

1. **Structural zeros** — individuals who will **never** experience the event (e.g. non-drivers in a car accident study, non-smokers in a lung cancer study)
2. **Sampling zeros** — individuals who *could* have the event but didn't in this observation window

**Zero-Inflated models** use a mixture of two components:
```
P(Y = 0) = π + (1-π) × P_count(Y = 0)   ← mixture of "always zero" + "count model zero"
P(Y = k) = (1-π) × P_count(Y = k)         ← only from count component, for k > 0
```

- **π** = probability of being in the "structural zero" group (modeled with logistic regression)
- **P_count** = Poisson or Negative Binomial for the count part

**ZINB** (Zero-Inflated Negative Binomial) handles both overdispersion AND excess zeros.

In [ ]:
# Fit Zero-Inflated Negative Binomial (ZINB)
X_count = sm.add_constant(insurance[['age','male','smoker','urban']])
X_infl  = sm.add_constant(insurance[['age','smoker']])  # zero-inflation predictors

zinb_model = ZeroInflatedNegativeBinomialP(
    insurance['claims'],
    X_count,
    exog_infl=X_infl,
    p=2  # NB-2 parameterization (Var = mu + alpha*mu^2)
).fit(method='bfgs', maxiter=200, disp=False)

print(zinb_model.summary())

In [ ]:
# Extract predicted zero probabilities vs actual zeros
predicted_zeros = zinb_model.predict(which='prob-zero')
actual_zero     = (insurance['claims'] == 0).astype(int)

print("=== Zero Prediction Quality ===")
print(f"  Actual zero rate:            {actual_zero.mean():.3f}")
print(f"  Predicted zero prob (mean):  {predicted_zeros.mean():.3f}")

# Compare all three models: Poisson, NB, ZINB
pois_zeros = stats.poisson.pmf(0, ins_pois.predict()).mean()
nb_zeros   = (nb_model.predict() / (nb_model.predict() + 1/nb_model.params['alpha'])
              ** (1/nb_model.params['alpha'])).mean()

fig, ax = plt.subplots(figsize=(9, 4))
models_z = ['Actual', 'Poisson', 'Neg Binomial', 'ZINB']
zero_rates = [actual_zero.mean(), pois_zeros, nb_zeros, predicted_zeros.mean()]
colors_z   = ['#1a1d23', '#e85d20', '#1e5fa8', '#1a7a45']
bars = ax.bar(models_z, zero_rates, color=colors_z, edgecolor='white', width=0.5)
for bar, val in zip(bars, zero_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Predicted Zero Rate')
ax.set_title('Zero Prediction: Poisson vs NB vs ZINB\n(ZINB best matches the actual zero rate)')
plt.tight_layout()
plt.show()

print("\n=== AIC Comparison (lower = better fit) ===")
try:
    print(f"  Poisson:         {ins_pois.aic:.2f}")
    print(f"  Neg Binomial:    {nb_model.aic:.2f}")
    print(f"  ZINB:            {zinb_model.aic:.2f}  {'✓ Best' if zinb_model.aic == min(ins_pois.aic, nb_model.aic, zinb_model.aic) else ''}")
except:
    print("  AIC comparison — see individual summaries above")

## 🗺️ Part 6 — Choosing the Right Count Model

```
Start here: Is your outcome a non-negative integer (count)?
         ↓ Yes
Is the mean ≈ variance?
    ↓ Yes              ↓ No (Var >> Mean)
  POISSON          Are there excess zeros?
                     ↓ Yes              ↓ No
                     ZIP/ZINB        NEG BINOMIAL
```

**Decision checklist:**

| Question | Test / Check | If Yes → |
|----------|-------------|----------|
| Is Var >> Mean? | Pearson χ²/df >> 1 | Use Negative Binomial |
| Excess zeros? | Observed zeros >> Poisson expected | Use ZIP or ZINB |
| Both? | Overdispersed AND excess zeros | Use ZINB |
| Exposure varies? | Different time periods / areas | Add log(exposure) as offset |

**Quick diagnostics in Python:**
- `pearson_resid²/df` → check for overdispersion
- `compare observed vs model-predicted P(Y=0)` → check for zero inflation
- `AIC` comparison across models → formal model selection

In [ ]:
# Full diagnostic comparison plot
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Observed vs predicted distribution
k = np.arange(0, 12)
obs_dist  = [(insurance['claims'] == ki).mean() for ki in k]
pois_dist = [stats.poisson.pmf(ki, ins_pois.predict()).mean() for ki in k]

axes[0,0].bar(k - 0.2, obs_dist,  0.35, label='Observed',  color='#1a1d23', alpha=0.8)
axes[0,0].bar(k + 0.2, pois_dist, 0.35, label='Poisson',   color='#e85d20', alpha=0.7)
axes[0,0].set_xlabel('Count'); axes[0,0].set_ylabel('Proportion')
axes[0,0].set_title('Observed vs Poisson Predicted Distribution')
axes[0,0].legend(); axes[0,0].set_xlim(-0.5, 11.5)

# 2. Pearson residuals by fitted value
axes[0,1].scatter(ins_pois.predict(), ins_pois.resid_pearson,
                  alpha=0.4, s=15, color='#e85d20')
axes[0,1].axhline(0,  color='black', lw=1, ls='--')
axes[0,1].axhline(2,  color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[0,1].axhline(-2, color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[0,1].set_xlabel('Fitted Values (Poisson)')
axes[0,1].set_ylabel('Pearson Residuals')
axes[0,1].set_title('Poisson Residuals — funnel shape = overdispersion')

# 3. NB residuals
nb_pearson = (insurance['claims'] - nb_model.predict()) / np.sqrt(nb_model.predict())
axes[1,0].scatter(nb_model.predict(), nb_pearson, alpha=0.4, s=15, color='#1e5fa8')
axes[1,0].axhline(0,  color='black', lw=1, ls='--')
axes[1,0].axhline(2,  color='#1e5fa8', lw=1, ls=':', alpha=0.7)
axes[1,0].axhline(-2, color='#1e5fa8', lw=1, ls=':', alpha=0.7)
axes[1,0].set_xlabel('Fitted Values (Neg Binomial)')
axes[1,0].set_ylabel('Approx Pearson Residuals')
axes[1,0].set_title('NB Residuals — better behaved than Poisson')

# 4. Rate ratio visualization
coef_nb   = nb_model.params.drop(['Intercept','alpha'])
irr_nb    = np.exp(coef_nb)
ci_nb     = np.exp(nb_model.conf_int().drop(['Intercept','alpha']))
colors_nb = ['#1a7a45' if v > 1 else '#e85d20' for v in irr_nb]

axes[1,1].barh(list(irr_nb.index), irr_nb.values, color=colors_nb, alpha=0.7,
               edgecolor='white', height=0.5)
axes[1,1].errorbar(irr_nb.values, range(len(irr_nb)),
                   xerr=[irr_nb.values - ci_nb[0].values,
                         ci_nb[1].values - irr_nb.values],
                   fmt='none', color='#1a1d23', capsize=4, lw=1.5)
axes[1,1].axvline(1, color='black', lw=1.5, ls='--')
axes[1,1].set_xlabel('Incidence Rate Ratio (IRR)')
axes[1,1].set_title('Negative Binomial IRR\n(green = increases count, red = decreases)')

plt.suptitle('Count Model Diagnostics — Insurance Claims', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 💼 Exercise

**Task 1:** Using the bike rentals dataset, fit a Poisson regression predicting `count` from all available predictors. Check for overdispersion. If present, refit as Negative Binomial. Does the AIC improve?

**Task 2:** Interpret the IRR for `workingday`. In plain English: how much more or less are bikes rented on working days vs non-working days, holding everything else constant?

**Task 3:** The insurance claims dataset has both overdispersion AND excess zeros. Compare Poisson, NB, and ZINB by AIC. Which fits best? Visualize the observed vs predicted count distribution for the winning model.

**Task 4:** Add `hour` as a predictor to the bike rental Poisson model. Is the hour of day significant? Plot the predicted count by hour (0-23) holding other variables at their mean.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────

# Task 1: Poisson → NB on bike rentals
# YOUR CODE HERE

# Task 2: Interpret IRR for workingday
# YOUR CODE HERE

# Task 3: Poisson vs NB vs ZINB on insurance — AIC comparison + distribution plot
# YOUR CODE HERE

# Task 4: Hour effect on bike rentals
# YOUR CODE HERE

In [ ]:
# ── Submit your results ─────────────────────────────────────────────────────
GITHUB_USERNAME = ""   # ← your GitHub username

answers = {
    "q1": "",  # What does the Poisson assumption Var(Y) = E(Y) mean, and how do you test if it's violated?
    "q2": "",  # How do you interpret an IRR of 1.45 for a binary predictor 'smoker'?
    "q3": "",  # What is the difference between structural zeros and sampling zeros?
    "q4": "",  # When would you choose ZINB over Negative Binomial?
    "q5": "",  # Why are Poisson standard errors too small when data is overdispersed?
}

missing = [k for k,v in answers.items() if not str(v).strip()]
n_answered = len(answers) - len(missing)

print(f"{'='*50}")
print(f"Quiz: {n_answered}/{len(answers)} answered")
if not missing:
    print("✅ All answered! Save: File → Save a copy in GitHub")
else:
    print(f"❌ Still empty: {missing}")
print(f"{'='*50}")

## 📚 Further Reading

| Resource | What it covers |
|----------|---------------|
| [ISLP Ch 4.6](https://intro-stat-learning.github.io/ISLP/) | Poisson regression — count outcomes |
| [statsmodels GLM docs](https://www.statsmodels.org/stable/glm.html) | Full GLM reference including Poisson & NB |
| [Cameron & Trivedi — Regression Analysis of Count Data](https://www.cambridge.org/core/books/regression-analysis-of-count-data/BBD7380C37A0FBF47E09D5E8EEDE14DC) | The definitive textbook on count models |
| [Model Explainability notebook](model-explainability.ipynb) | SHAP for GLMs and count models |
| [Logistic Regression notebook](logistic-regression.ipynb) | Binary outcome — the classification analogue |

---
---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*